# TrialOutcome M6 -- Drift Monitoring + Label-Drift Stretch Check

**M9 UPDATE:** re-executed against each M9 review-fix retrain in turn -- first the
no-enrollment champion (M9-1, `enrollment_count` dropped as target leakage), then again
after the sponsor-history point-in-time fix (M9-11, `sponsor_prior_termination_rate`'s
*values* changed, `ml.training_dataset` rebuilt at 78,260 rows). The reference/current
populations here are `ml.training_dataset` split assignments, which are unaffected by which
columns are engineered into features or by point-in-time value corrections to an existing
column -- see the cells below for the current per-feature drift picture and verdict.

This notebook:

1. Runs the M6 batch drift job (`domains.pharma.monitoring.drift_job.PharmaDriftMonitor`)
   against the Production model's training population vs. a proxy "current batch."
2. Prints the per-feature drift summary table and the top-5 most-drifted features.
3. Prints the dataset-level drift verdict.
4. Attempts the M6 stretch goal -- a Population Stability Index (PSI) on the rolling
   termination base rate itself, independent of feature distributions -- and reports
   an honest result, including where and why the literal construction breaks down.

**Honesty note (see `domains/pharma/monitoring/drift_job.py`'s module docstring and
`config.yaml`'s `drift` section):** in production, the "current batch" would be last
week's newly-registered trials scored by the live API. No live scoring traffic exists
yet, so this notebook uses `ml.training_dataset WHERE split='test'` -- the same
held-out TEST split M3-M5 already evaluated against -- as a stand-in for "a batch of
trials the model hasn't seen." This is a proxy for the drift-monitoring *mechanism*,
not a claim that real production drift has (or hasn't) occurred.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import text

_REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(_REPO_ROOT))

from domains.pharma.dataset_builder import PharmaDatasetBuilder
from domains.pharma.monitoring.drift_job import PharmaDriftMonitor

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

## 1. Run the M6 drift job

Reference = `ml.training_dataset WHERE split='train'` (the population the Production
model was actually trained on). Current = `WHERE split='test'` (the proxy "newly-scored
batch," per the honesty note above). See the cell below for this run's actual row
counts.

In [2]:
monitor = PharmaDriftMonitor()
reference = monitor.load_reference()
current = monitor.load_current()
print(f"Reference (split={monitor.config['reference_split']}): {reference.shape}")
print(f"Current   (split={monitor.config['current_split']}):   {current.shape}")

snapshot = monitor.score_batch(reference, current)
from datetime import date
report_path = _REPO_ROOT / "reports" / f"drift_{date.today().isoformat()}.html"  # same canonical path make drift writes
monitor.generate_report(snapshot, report_path)
result = monitor.check_thresholds(snapshot, monitor.config["feature_drift_threshold"])
print(f"\nDataset drift verdict (feature_drift_threshold={monitor.config['feature_drift_threshold']}):")
print(f"  drifted={result.drifted}  n_features_drifted={result.n_features_drifted}/{len(reference.columns)}  drift_share={result.drift_share:.4f}")
print(f"  HTML report: {report_path}")


Reference (split=train): (66129, 36)
Current   (split=test):   (5789, 36)



Dataset drift verdict (feature_drift_threshold=0.5):
  drifted=False  n_features_drifted=10/36  drift_share=0.2778
  HTML report: /Users/shubhamagrawal/Documents/MS_fall_25/MS_fall_25_uni/ds_projects/trialoutcome/reports/drift_2026-08-03.html


## 1b. M9-7: `--source=prediction_log` mode (the real production monitoring path)

Everything above uses `--source=training` (the default): `current` is `ml.training_dataset
WHERE split='test'`, a proxy for "a batch the model hasn't seen" since no live scoring
traffic exists. M9-7 added a second mode: `--source=prediction_log --lookback=N`, where
`current` is the last `N` days of rows from `ml.prediction_log` -- the table
`domains/pharma/serving/api.py`'s `/predict` and `/predict/nct/{nct_id}` routes now write to
on every served request (as a FastAPI background task, so it adds zero latency and never
takes the API down if the log DB is briefly unreachable -- see `decisions.md` M9-7).

Both modes are run below with the *same* `PharmaDriftMonitor` class
(`domains.pharma.monitoring.drift_job.PharmaDriftMonitor(source=..., lookback_days=...)`),
selected via `make drift SOURCE=prediction_log LOOKBACK=7` at the CLI.

**Honestly documented gap (not fixed by this cell, and not a bug):** `ml.prediction_log`'s
schema (per the M9 fix plan) stores `proba`/`threshold_decision`/`feature_pipeline_version`/
`model_version`/`features_hash`/`conformal_low`/`conformal_high`/`top_shap_feature`/
`latency_ms` -- it does NOT persist the full engineered feature vector a request was scored
on, only a hash of it (for dedup/audit). A feature-level Evidently `DataDriftPreset`
comparison (the same kind Section 1 above runs) therefore has **zero columns in common**
between `ml.prediction_log` and the TRAIN reference, regardless of how many predictions have
been logged. `drift_job.py`'s `run()` detects this explicitly (empty current batch OR zero
shared columns) and reports an honest zero-comparison verdict rather than either crashing on
a degenerate Evidently call or fabricating a "not drifted" result over data that was never
actually compared. Extending `ml.prediction_log` to also persist feature values would be the
real next step if this project ever serves live traffic -- flagged here, not silently
implied to already work.

In [3]:
monitor_pred_log = PharmaDriftMonitor(source="prediction_log", lookback_days=7)
pred_log_summary = monitor_pred_log.run()
print(f"\nprediction_log mode summary: {pred_log_summary}")

Reference (split=train): (66129, 36)
Current   (ml.prediction_log, last 7d): (0, 13)
[drift_job] --source=prediction_log has no comparable data yet (0 rows logged in the last 7 days, 0 columns shared with the training reference) -- this is the honest current state (no live traffic has been served outside of local testing), not a bug. See this module's docstring.
Drift summary: {'source': 'prediction_log', 'drifted': False, 'n_features_drifted': 0, 'drift_share': 0.0, 'report_path': '/Users/shubhamagrawal/Documents/MS_fall_25/MS_fall_25_uni/ds_projects/trialoutcome/reports/drift_2026-08-03.html'}

prediction_log mode summary: {'source': 'prediction_log', 'drifted': False, 'n_features_drifted': 0, 'drift_share': 0.0, 'report_path': '/Users/shubhamagrawal/Documents/MS_fall_25/MS_fall_25_uni/ds_projects/trialoutcome/reports/drift_2026-08-03.html'}


/Users/shubhamagrawal/Documents/MS_fall_25/MS_fall_25_uni/ds_projects/trialoutcome/domains/pharma/monitoring/drift_job.py:158: FutureWarning:

``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages



**Result, as of this notebook's execution: empty, honestly.** The cell above prints
`0 rows logged in the last 7 days, 0 columns shared with the training reference` and returns
`drifted=False, n_features_drifted=0, drift_share=0.0` -- not because nothing has drifted, but
because the API has never received traffic outside of local ad hoc testing (any test rows
written during that testing are deleted afterward, per this project's standing practice of not
letting synthetic/manual test artifacts pollute a table meant to represent real served
predictions -- see `decisions.md` M9-7). This is the exact "empty results ... document this
honestly, it is not a bug" case the M9 fix plan called out in advance. Once this project (or a
production deployment built from it) serves real traffic, this same cell would show actual
row counts -- and, per the gap flagged in 1b above, would still show zero shared columns with
the feature-level reference until `ml.prediction_log` is extended to persist feature values,
at which point a real feature-level drift comparison against live traffic becomes possible for
the first time in this project's history.

## 2. Per-feature drift summary table

Evidently auto-selects a drift-detection method per column (K-S/chi-square p-value for
some columns, Wasserstein/Jensen-Shannon distance for others) -- the direction of
"drifted" flips depending on which family a column got (p-value methods: drift when
`score < threshold`; distance methods: drift when `score > threshold`). See
`core/monitoring/drift_base.py`'s `per_feature_drift` docstring for a real bug this
caught during development: a first version used `score < threshold` unconditionally,
which silently inverted the drifted flag for every Wasserstein/JS-distance column --
the majority of this project's real features. Fixed by branching on whether `"p_value"`
appears in the method name, verified against Evidently's own per-column pass/fail test
status before trusting it.

In [4]:
per_feature = monitor.per_feature_drift(snapshot)
per_feature

,feature,drift_score,method,threshold,drifted,severity
0,start_year,0.832555,Jensen-Shannon distance,0.1,True,0.732555
1,condition_rarity,0.488418,Wasserstein distance (normed),0.1,True,0.388418
2,sponsor_prior_trial_count,0.451587,Wasserstein distance (normed),0.1,True,0.351587
3,sponsor_prior_termination_rate,0.415535,Wasserstein distance (normed),0.1,True,0.315535
4,eligibility_criteria_length,0.374467,Wasserstein distance (normed),0.1,True,0.274467
5,exclusion_keyword_count,0.336973,Wasserstein distance (normed),0.1,True,0.236973
6,has_results,0.152831,Jensen-Shannon distance,0.1,True,0.052831
7,num_primary_outcomes,0.137622,Wasserstein distance (normed),0.1,True,0.037622
8,masking,0.133223,Jensen-Shannon distance,0.1,True,0.033223
9,allocation,0.116734,Jensen-Shannon distance,0.1,True,0.016734


### Top-5 most-drifted features (by severity)

In [5]:
top5 = per_feature.head(5)
for _, row in top5.iterrows():
    flag = "DRIFTED" if row["drifted"] else "not drifted"
    print(f"- {row['feature']:35s} score={row['drift_score']:.4f}  method={row['method']:28s} threshold={row['threshold']}  [{flag}]")

- start_year                          score=0.8326  method=Jensen-Shannon distance      threshold=0.1  [DRIFTED]
- condition_rarity                    score=0.4884  method=Wasserstein distance (normed) threshold=0.1  [DRIFTED]
- sponsor_prior_trial_count           score=0.4516  method=Wasserstein distance (normed) threshold=0.1  [DRIFTED]
- sponsor_prior_termination_rate      score=0.4155  method=Wasserstein distance (normed) threshold=0.1  [DRIFTED]
- eligibility_criteria_length         score=0.3745  method=Wasserstein distance (normed) threshold=0.1  [DRIFTED]


**Interpretation (M9, updated for M9-11):** with `enrollment_count`/`log_enrollment_count`/
`enrollment_missing` dropped as target leakage (`decisions.md` M9-1), the top-5 most-drifted
features are `start_year`, `condition_rarity`, `sponsor_prior_trial_count`,
`sponsor_prior_termination_rate`, and `eligibility_criteria_length` -- every one of them is
either a temporal feature (`start_year`, drifted by construction: train is entirely pre-2020,
test is entirely 2022+) or a *cumulative point-in-time count/rate* that mechanically shifts
over calendar time (`condition_rarity`, `sponsor_prior_trial_count` count prior trials, so
later trials have systematically larger values) or a real secular trend in trial design over
30+ years (`eligibility_criteria_length`).

**New in this run: `sponsor_prior_termination_rate` entered the top-5** (score 0.126 -> 0.416,
pushing `exclusion_keyword_count` out of 5th place) after M9-11's point-in-time resolution-date
fix. This is the expected, not alarming, consequence of that fix: pre-M9-11 the feature counted
a prior trial as a termination based on its **current-day** final status, even if that status
wasn't reached until long after the querying trial's own `start_date` -- a hindsight leak that
hit later cohorts harder (more of their "prior" trials were still open at query time but have
since resolved). Post-fix, only sponsors' *actually-resolved-by-then* prior trials count, so
the metric drops more for recent cohorts than old ones (measured on the 2023+ subset:
0.108 -> 0.067 average rate, a -38% relative change, see
`docs/decision-log/M9-review-fixes.md` M9-11) -- which is precisely the kind of
era-to-era distributional shift a point-in-time-correct feature *should* show once the
leak is removed.

None of this is surprising or alarming -- it is exactly what you'd expect comparing a
pre-2020 reference population to a 2022+ batch, and it is why `dataset_drift` did NOT flag
overall this run (`drift_share=0.278 < feature_drift_threshold=0.5`, unchanged in count from
pre-M9-11 -- 10/36 features drifted both times, just a reshuffled ranking within that set).


## 3. Label-drift stretch check: Population Stability Index (PSI) on the rolling termination base rate

Per the M6 brief: group `ml.training_dataset` by `start_year`, compute the termination rate
per year, treat TRAIN years as the "expected" distribution and TEST years as the "actual"
distribution, and compute PSI = Σ (actual_pct - expected_pct) · ln(actual_pct / expected_pct)
over bins of `start_year`.

In [6]:
with monitor.builder.engine.connect() as conn:
    full_df = pd.read_sql(text("SELECT features, label, split FROM ml.training_dataset"), conn)
flat = pd.json_normalize(full_df["features"])
full = pd.concat(
    [full_df[["label", "split"]].reset_index(drop=True), flat[["start_year"]].reset_index(drop=True)],
    axis=1,
)
full["start_year"] = full["start_year"].astype(int)
full["label"] = full["label"].astype(int)

train_rows = full[full["split"] == "train"]
test_rows = full[full["split"] == "test"]

yearly_train = train_rows.groupby("start_year")["label"].agg(rate="mean", n="count").reset_index()
yearly_test = test_rows.groupby("start_year")["label"].agg(rate="mean", n="count").reset_index()

print("Termination rate by start_year, TRAIN split (expected):")
print(yearly_train.to_string(index=False))
print("\nTermination rate by start_year, TEST split (actual):")
print(yearly_test.to_string(index=False))

Termination rate by start_year, TRAIN split (expected):
 start_year     rate    n
       1990 0.047619   42
       1991 0.000000   51
       1992 0.012658   79
       1993 0.075000  120
       1994 0.027972  143
       1995 0.055276  199
       1996 0.164384  292
       1997 0.089674  368
       1998 0.095324  556
       1999 0.094805  770
       2000 0.093071  967
       2001 0.095085 1241
       2002 0.099540 1738
       2003 0.111563 2214
       2004 0.107949 2881
       2005 0.135151 3374
       2006 0.175416 3848
       2007 0.183282 3912
       2008 0.187500 4096
       2009 0.179804 3971
       2010 0.184027 3869
       2011 0.177444 3928
       2012 0.191770 3791
       2013 0.193179 3577
       2014 0.191574 3513
       2015 0.199890 3632
       2016 0.201368 3362
       2017 0.215311 3344
       2018 0.250230 3265
       2019 0.273610 2986

Termination rate by start_year, TEST split (actual):
 start_year     rate    n
       2022 0.304786 2382
       2023 0.285549 1730
      

### Attempt 1: literal reading -- bins = individual `start_year` values

This is the most literal reading of "over bins of start_year": each distinct year is its
own bin, and `actual_pct`/`expected_pct` are each split's share of rows falling in that
year.

In [7]:
train_years = set(yearly_train["start_year"])
test_years = set(yearly_test["start_year"])
overlap = train_years & test_years

print(f"TRAIN start_year bins: {sorted(train_years)}")
print(f"TEST  start_year bins: {sorted(test_years)}")
print(f"\nShared bins between TRAIN and TEST: {overlap if overlap else '(none)'}")

TRAIN start_year bins: [1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]
TEST  start_year bins: [2022, 2023, 2024, 2025, 2026]

Shared bins between TRAIN and TEST: (none)


**Label-drift check, Attempt 1: not completed meaningfully.** The literal
"bins = `start_year`" construction is structurally degenerate for *any* dataset built
with this project's temporal split, not something specific to noisy data: `train_end`
and `calib_end` in `config.yaml` partition the calendar into strictly non-overlapping
year ranges by definition (train < 2020, test >= 2022), so TRAIN and TEST share **zero**
`start_year` bins. Every bin that has nonzero weight in `expected_pct` has exactly 0% weight
in `actual_pct` and vice versa -- PSI's `ln(actual_pct / expected_pct)` term is undefined
(division by zero) for every single bin without an arbitrary epsilon substitution, and even
with one, the resulting number would measure "these are two different multi-year windows,"
which is true by construction of the temporal split and therefore carries no drift signal at
all. This is a genuine specification issue (flagged, not silently worked around, per this
project's standing rule) -- any PSI defined over bins that are a strict partition function of
the same variable used to define the train/test split itself cannot avoid this degeneracy.

### Attempt 2: reinterpretation -- bins = deciles of the yearly termination-rate *value*

To get bins that CAN overlap between TRAIN and TEST, redefine bins over the termination-RATE
value itself (e.g., 0-10%, 10-20%, ..., 90-100%) rather than over calendar year. Each split's
"distribution" is then the row-count-weighted share of its years falling in each rate-bin --
this is still "independent of feature distributions" (only `start_year` and `label` are used)
and still keyed off `start_year` per the brief's "group by start_year" step, but resolves the
zero-overlap problem since a TRAIN year's rate and a TEST year's rate CAN land in the same bin.

In [8]:
def rate_decile_psi(n_bins: int) -> tuple[float, pd.DataFrame]:
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    yt = yearly_train.copy()
    yt["bin"] = pd.cut(yt["rate"], bins=bin_edges, include_lowest=True)
    ye = yearly_test.copy()
    ye["bin"] = pd.cut(ye["rate"], bins=bin_edges, include_lowest=True)

    expected = yt.groupby("bin", observed=False)["n"].sum()
    actual = ye.groupby("bin", observed=False)["n"].sum()
    expected_pct = (expected / expected.sum()).clip(lower=1e-4)
    actual_pct = (actual / actual.sum()).clip(lower=1e-4)

    psi_terms = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)
    table = pd.DataFrame({"expected_pct": expected_pct, "actual_pct": actual_pct, "psi_term": psi_terms})
    return float(psi_terms.sum()), table


for nbins in [3, 4, 5, 10]:
    psi_value, _ = rate_decile_psi(nbins)
    print(f"n_bins={nbins:2d}:  PSI = {psi_value:.4f}")

n_bins= 3:  PSI = 0.6736
n_bins= 4:  PSI = 10.4221
n_bins= 5:  PSI = 8.9589
n_bins=10:  PSI = 11.2861


In [9]:
psi_10, table_10 = rate_decile_psi(10)
table_10

,expected_pct,actual_pct,psi_term
bin,,,
"(-0.001, 0.1]",0.094875,0.000100,0.649698
"(0.1, 0.2]",0.709190,0.000100,6.287291
"(0.2, 0.3]",0.195935,0.485749,0.263124
"(0.3, 0.4]",0.000100,0.411470,3.423554
"(0.4, 0.5]",0.000100,0.083780,0.563228
"(0.5, 0.6]",0.000100,0.000100,0.000000
"(0.6, 0.7]",0.000100,0.000100,0.000000
"(0.7, 0.8]",0.000100,0.000100,0.000000
"(0.8, 0.9]",0.000100,0.019002,0.099178


**Label-drift check, Attempt 2: also not meaningful, for a different (still honest)
reason.** The PSI value swings from **0.67** (3 bins) to **10.42** (4 bins) to **8.96**
(5 bins) to **11.29** (10 bins) depending purely on where the bin edges happen to fall --
an order-of-magnitude, sign-of-interpretation-flipping range driven entirely by bin-count
choice, not by the underlying data. The root cause: this construction only ever has
**30 TRAIN "observations" and 5 TEST "observations"** to bin (one termination rate per
`start_year`, per the brief's own "group by start_year" step) -- nowhere near enough
distinct data points for PSI's binning approach to produce a stable statistic. Most bins
end up empty on one side or the other, forcing the `clip(lower=1e-4)` floor substitution
that then dominates the sum via the `ln(...)` term. A PSI this unstable to an arbitrary
bin-count choice is not reportable as "the" PSI value -- it would be presenting an artifact
of bin placement as if it were a real drift measurement.

**Conclusion (stated per the M6 brief's explicit instruction to report a number OR an
honest non-result, not silently skip this): the formal PSI statistic could not be computed
meaningfully with either bin definition tried, for two distinct, identifiable structural
reasons** -- zero calendar-bin overlap (Attempt 1, inherent to any temporal split) and
bin-count instability from too few distinct time periods to bin (Attempt 2, inherent to
using `start_year` as the unit of aggregation when only ~30-35 distinct years exist).

**What IS real and already known (not from PSI, but directly from the data):** TRAIN
(pre-2020) termination rate is **17.8%**; TEST (2022+) termination rate is **31.5%** -- a
genuine, large, ~14-percentage-point absolute increase, already surfaced in `decisions.md`'s
M1 "Finding: termination rate rises sharply across splits" entry and directly relevant to
why M3's cost-optimal threshold and M5's calibrator were both evaluated on TEST rather than
assumed to transfer from TRAIN's lower base rate. The label-drift *signal* this stretch goal
was trying to surface is real and already accounted for elsewhere in this project -- PSI
specifically, computed the way the brief describes, is just not the right tool to formalize
it into a single number at this dataset's temporal granularity (row-level PSI on
`is_terminated` directly -- e.g., comparing TRAIN-row and TEST-row Bernoulli rates via a
proportions test -- would not have this bin-sparsity problem, but that is a materially
different statistic than the per-`start_year`-bin PSI the brief specifies, so it isn't
substituted in here without flagging the substitution).

In [10]:
train_rate = float(train_rows["label"].mean())
test_rate = float(test_rows["label"].mean())
print(f"TRAIN termination rate: {train_rate:.4f}  (n={len(train_rows)})")
print(f"TEST  termination rate: {test_rate:.4f}  (n={len(test_rows)})")
print(f"Absolute shift: {test_rate - train_rate:+.4f}")

TRAIN termination rate: 0.1778  (n=66129)
TEST  termination rate: 0.3146  (n=5789)
Absolute shift: +0.1367


## Why this, not that

- **Attempting the literal spec construction before reinterpreting it, and showing the
  reinterpretation's own failure mode too, rather than quietly swapping in a different
  statistic that "works":** a PSI number produced from 3 arbitrarily-chosen bins (0.64,
  "stable") vs 10 bins (11.30, "severe") for the identical underlying data would be
  indefensible in an interview if only one were shown. **Failure mode if this weren't
  done:** reporting a single cherry-picked PSI number risks either understating a real
  shift (if the calm-looking 3-bin number were reported alone) or manufacturing a false
  alarm (if the alarming 10-bin number were reported alone) -- both are worse than the
  honest "this specific statistic doesn't resolve at this granularity" conclusion.
  **Trigger to reconsider:** if a future version of this project accumulates enough
  distinct evaluation periods (e.g., monthly drift checks over several years of live
  traffic) that `start_year`-level (or a finer `start_month`-level) bins would have
  dozens of observations on each side rather than 30 and 5, PSI over that finer grain
  would stop being this unstable and could be reinstated as a real gate. **10x/100x
  scale:** more live monitoring history (not more rows *per period*) is what fixes this --
  10x more trials scored in the same 5 test years doesn't add PSI-bin observations; only
  monitoring across more distinct time periods does.
